# 10 - Validation-frozen tiers and leakage-safe prediction diagnostic

Recomputes class tiers from validation data and rebuilds the baseline diagnostic with validation-only covariates, test-only compression-loss targets, and attack-family-held-out evaluation.

**Safety:** this notebook writes only new files under `results/tables/comnet/` and does not overwrite archived manuscript result tables.

In [ ]:
# Colab/bootstrap cell: no tokens or credentials are required.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import os, sys, json, time
from pathlib import Path

REPO = Path('/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression')
if not REPO.exists():
    # Local/Jupyter fallback: run the notebook from the repository root.
    REPO = Path.cwd()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.config import CFG, PATHS, set_all_seeds
set_all_seeds(CFG['anchor_seed'])

OUT_TABLE = PATHS.tables('comnet')
OUT_TABLE.mkdir(parents=True, exist_ok=True)
print('Repository:', REPO)
print('Outputs:', OUT_TABLE)


## Configuration

In [ ]:
DATASET = 'ciciot2023'
ARCH = 'cnn1d'
ARCH_KW = {'channels': (64, 128)}
SEEDS = list(CFG['seeds'])
ANCHOR_SEED = int(CFG['anchor_seed'])
CELLS = ['prune50', 'prune80', 'distillation']
USE_ARCHIVED_TARGET_IF_CHECKPOINT_MISSING = True
ARCHITECTURE_GATE_SPECS = {
    'cnn1d': {'channels': (64, 128)},
    'mlp': {'hidden': (256, 128)},
}
# Add the exact archived transformer kwargs here only when they are known.
# Do not guess a transformer configuration merely to complete the table.


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import f1_score, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut

from src.data import load_raw, clean, temporal_within_capture_split
from src.train import load_anchor, predict, per_class_recall_table
from src import predict as predmod
from src.comnet_audit import assign_validation_tiers, infer_family

df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR_SEED), DATASET)
splits = temporal_within_capture_split(df, ANCHOR_SEED)


## Freeze class tiers from validation data only

In [ ]:
import glob
print('\n'.join(sorted(glob.glob(
    '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/models/ciciot2023/*'))))

In [ ]:
# --- Train and save baseline seeds 1-4 (needed by notebooks 10 and 11) ---
from google.colab import drive; drive.mount('/content/drive')
import os, sys, torch
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)

from src.config import CFG, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src.train import train_model

assert torch.cuda.is_available(), 'switch to GPU runtime first'
ANCHOR = int(CFG['anchor_seed'])
df = clean(load_raw('ciciot2023', subsample=True, seed=ANCHOR), 'ciciot2023')
splits = temporal_within_capture_split(df, seed=ANCHOR)   # frozen primary split for all seeds

for seed in [1, 2, 3, 4]:
    print(f'===== training baseline seed {seed} =====')
    train_model('cnn1d', df, 'ciciot2023', splits, seed,
                arch_kwargs={'channels': (64, 128)}, save=True, verbose=True)
print('all four baseline checkpoints saved')

In [ ]:
val_recall = {}
val_macro = []
models = {}
for seed in SEEDS:
    m, le_s, scaler_s, feat_s = load_anchor(DATASET, ARCH, 'M0', seed, arch_kwargs=ARCH_KW)
    yt, yp, _ = predict(m, df, splits, le_s, scaler_s, feat_s, which='val')
    tab = per_class_recall_table(yt, yp, le_s).set_index('label')['recall']
    val_recall[seed] = tab
    val_macro.append({'arch': ARCH, 'seed': seed, 'validation_macro_f1': f1_score(yt, yp, average='macro')})
    models[seed] = (m, le_s, scaler_s, feat_s)

val_recall_df = pd.DataFrame(val_recall)
validation_tiers = assign_validation_tiers(val_recall_df)
validation_tiers.to_csv(OUT_TABLE / 'validation_defined_tiers.csv')
pd.DataFrame(val_macro).to_csv(OUT_TABLE / 'validation_architecture_gate_cnn.csv', index=False)

# Validation-only architecture gate. Missing checkpoints are recorded, not hidden.
gate_rows = []
for gate_arch, gate_kw in ARCHITECTURE_GATE_SPECS.items():
    for seed in SEEDS:
        try:
            gm, gle, gsc, gfeat = load_anchor(DATASET, gate_arch, 'M0', seed, arch_kwargs=gate_kw)
            gy, gp, _ = predict(gm, df, splits, gle, gsc, gfeat, which='val')
            gate_rows.append({'arch':gate_arch, 'seed':seed,
                              'validation_macro_f1':f1_score(gy,gp,average='macro'),
                              'status':'ok'})
        except Exception as exc:
            gate_rows.append({'arch':gate_arch, 'seed':seed,
                              'validation_macro_f1':np.nan,
                              'status':f'{type(exc).__name__}: {exc}'})
architecture_gate = pd.DataFrame(gate_rows)
architecture_gate.to_csv(OUT_TABLE / 'validation_architecture_gate.csv', index=False)
display(architecture_gate)

tier_sensitivity = []
for floor in (0.05, 0.075, 0.10):
    for robust in (0.85, 0.90, 0.95):
        t = assign_validation_tiers(val_recall_df, floored_max=floor, robust_min=robust)
        counts = t['validation_tier'].value_counts().to_dict()
        tier_sensitivity.append({'floored_max':floor, 'robust_min':robust, **counts})
pd.DataFrame(tier_sensitivity).fillna(0).to_csv(
    OUT_TABLE / 'validation_tier_threshold_sensitivity.csv', index=False
)

display(validation_tiers)
print(validation_tiers['validation_tier'].value_counts())


## Build predictor covariates on validation only; reserve test for the compression-loss target

In [ ]:
m0, le, scaler, feat_cols = models[ANCHOR_SEED]
feature_df = predmod.assemble_features(
    m0, df, splits, scaler, feat_cols, le,
    which='val', max_n=60000, seed=ANCHOR_SEED,
)
feature_df.index.name = 'class'
feature_df['family'] = [infer_family(c) for c in feature_df.index]
feature_df.to_csv(OUT_TABLE / 'prediction_features_validation_only.csv')

# Test-set M0 recall defines the outcome scale, but never enters feature construction.
yt0, yp0, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
r0 = per_class_recall_table(yt0, yp0, le).set_index('label')['recall']
loss_targets = {}

for cell in CELLS:
    kw = {'channels': (24, 48)} if cell == 'distillation' else ARCH_KW
    try:
        mc, lec, scc, fc = load_anchor(DATASET, ARCH, cell, ANCHOR_SEED, arch_kwargs=kw)
        yt, yp, _ = predict(mc, df, splits, lec, scc, fc, which='test')
        rc = per_class_recall_table(yt, yp, lec).set_index('label')['recall']
        loss_targets[cell] = (r0 - rc).rename(cell)
        print('target from checkpoint:', cell)
    except Exception as exc:
        print('checkpoint unavailable:', cell, exc)

if USE_ARCHIVED_TARGET_IF_CHECKPOINT_MISSING and len(loss_targets) < len(CELLS):
    archived_path = PATHS.tables('compression', 'cnn1d_per_class_recall_matrix.csv')
    if archived_path.exists():
        archived = pd.read_csv(archived_path, index_col=0)
        archived.index.name = 'label'
        m0_col = next((c for c in ('M0','m0','baseline') if c in archived.columns), None)
        if m0_col is None:
            raise RuntimeError(f'Cannot identify M0 column in {archived_path}')
        for cell in CELLS:
            if cell not in loss_targets and cell in archived.columns:
                loss_targets[cell] = (archived[m0_col].astype(float) - archived[cell].astype(float)).rename(cell)
                print('target from archived result table:', cell)

if not loss_targets:
    raise RuntimeError('No test-set compression-loss targets were found.')
loss_df = pd.concat(loss_targets.values(), axis=1)
loss_df.to_csv(OUT_TABLE / 'test_recall_loss_targets.csv')

## Family-held-out diagnostic evaluation

In [ ]:
FEATURE_SETS = {
    'frequency_only': ['support'],
    'margin_only': ['margin'],
    'confusability': ['sim_nearest_higher_freq','confusion_offdiag','pred_entropy'],
    'geometry': ['etf_cos_to_others','margin'],
    'all_baseline': ['support','baseline_recall','etf_cos_to_others','margin',
                     'sim_nearest_higher_freq','confusion_offdiag','pred_entropy'],
}

def grouped_evaluate(table, cols):
    d = table.dropna(subset=cols + ['loss','family']).copy()
    X = d[cols].astype(float).to_numpy(); y = d['loss'].astype(float).to_numpy()
    groups = d['family'].to_numpy()
    logo = LeaveOneGroupOut()
    pred = np.full(len(d), np.nan)
    mean_pred = np.full(len(d), np.nan)
    for tr, te in logo.split(X, y, groups):
        if len(tr) < max(5, len(cols)+2):
            continue
        model = Pipeline([('scale', StandardScaler()), ('reg', LinearRegression())])
        model.fit(X[tr], y[tr]); pred[te] = model.predict(X[te])
        mean_pred[te] = np.mean(y[tr])
    ok = np.isfinite(pred)
    if ok.sum() < 5:
        return {'n': int(ok.sum()), 'n_families': int(pd.Series(groups).nunique()),
                'spearman': np.nan, 'r2': np.nan, 'mae': np.nan,
                'mean_baseline_r2': np.nan, 'mean_baseline_mae': np.nan}
    rho = spearmanr(pred[ok], y[ok]).statistic
    return {'n': int(ok.sum()), 'n_families': int(pd.Series(groups[ok]).nunique()),
            'spearman': float(rho),
            'r2': float(r2_score(y[ok], pred[ok])),
            'mae': float(mean_absolute_error(y[ok], pred[ok])),
            'mean_baseline_r2': float(r2_score(y[ok], mean_pred[ok])),
            'mean_baseline_mae': float(mean_absolute_error(y[ok], mean_pred[ok]))}

rows = []
for cell in loss_df.columns:
    table = feature_df.join(loss_df[[cell]].rename(columns={cell:'loss'}), how='inner')
    for name, cols in FEATURE_SETS.items():
        missing = [c for c in cols if c not in table.columns]
        if missing:
            rows.append({'cell':cell, 'model':name, 'error':'missing '+','.join(missing),
                         'features':'|'.join(cols)})
            continue
        out = grouped_evaluate(table, cols)
        rows.append({'cell': cell, 'model': name, **out, 'features': '|'.join(cols)})

safe_pred_results = pd.DataFrame(rows)
safe_pred_results.to_csv(OUT_TABLE / 'prediction_validation_features_family_heldout.csv', index=False)
display(safe_pred_results.round(4))


## Interpretation gate

In [ ]:
print('All predictor covariates in this notebook were built on validation data; compression-loss targets were measured on test.')
print('Do not restore a pre-deployment prediction claim unless family-held-out performance beats the mean and frequency baselines consistently.')
